<a href="https://colab.research.google.com/github/asoali67/Required-Assignment-17.1-Comparing-Classifiers/blob/main/prompt_III.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [28]:
import pandas as pd
import numpy as np # linear algebra
import matplotlib.pyplot as plt # data visualization
import seaborn as sns # statistical data visualization

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, precision_recall_curve, roc_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import category_encoders as ce

In [29]:
df = pd.read_csv('sample_data/bank-additional-full.csv', sep = ';')

In [30]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [32]:
# checking null values
print(df.isnull().sum())

col_names = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome','education']
for col in col_names:
    print(df[col].value_counts())
    print('====================')

# in feature 'default', the size of the unknown is large and its very unbalanced
# in feature 'poutcome' , largest value is nonexistent
# remove these two columns
df = df.drop(['default', 'poutcome'], axis=1)
print(df.info())

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64
housing
yes        21576
no         18622
unknown      990
Name: count

In [33]:
# explore the target variable
df['y'].value_counts()

,count
y,
no,36548
yes,4640


In [34]:
# encoding part
#i will use one OneHotEncoder for month and day_of_week, and category_encoders for rest
# reset column names
col_names = ['job', 'marital', 'housing', 'loan', 'contact', 'month', 'day_of_week']
ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
ohetransform = ohe.fit_transform(df[['month', 'day_of_week']])

df = pd.concat([df, ohetransform], axis=1).drop(columns=['month', 'day_of_week'])

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 32 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              41188 non-null  int64  
 1   job              41188 non-null  object 
 2   marital          41188 non-null  object 
 3   education        41188 non-null  object 
 4   housing          41188 non-null  object 
 5   loan             41188 non-null  object 
 6   contact          41188 non-null  object 
 7   duration         41188 non-null  int64  
 8   campaign         41188 non-null  int64  
 9   pdays            41188 non-null  int64  
 10  previous         41188 non-null  int64  
 11  emp.var.rate     41188 non-null  float64
 12  cons.price.idx   41188 non-null  float64
 13  cons.conf.idx    41188 non-null  float64
 14  euribor3m        41188 non-null  float64
 15  nr.employed      41188 non-null  float64
 16  y                41188 non-null  object 
 17  month_apr   

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [35]:
X = df.drop(['y'], axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

In [36]:
# check the shape of X_train and X_test

X_train.shape, X_test.shape

((27595, 31), (13593, 31))

In [37]:


encoder = ce.OrdinalEncoder(cols=['job', 'marital', 'housing', 'loan', 'contact', 'education'])
X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)

print(X_test.info())
print(X_test.head())

<class 'pandas.core.frame.DataFrame'>
Index: 13593 entries, 32884 to 25748
Data columns (total 31 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              13593 non-null  int64  
 1   job              13593 non-null  int64  
 2   marital          13593 non-null  int64  
 3   education        13593 non-null  int64  
 4   housing          13593 non-null  int64  
 5   loan             13593 non-null  int64  
 6   contact          13593 non-null  int64  
 7   duration         13593 non-null  int64  
 8   campaign         13593 non-null  int64  
 9   pdays            13593 non-null  int64  
 10  previous         13593 non-null  int64  
 11  emp.var.rate     13593 non-null  float64
 12  cons.price.idx   13593 non-null  float64
 13  cons.conf.idx    13593 non-null  float64
 14  euribor3m        13593 non-null  float64
 15  nr.employed      13593 non-null  float64
 16  month_apr        13593 non-null  float64
 17  month_aug    

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [38]:
dummy_clf = ''
baseline_score = ''

dummy_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('dummy_model', DummyClassifier())
])

param_grid = {
    'scaler__with_mean': [True, False],                 # Optimizes the preprocessing step
    'dummy_model__strategy': ['most_frequent', 'stratified', 'prior']
}

grid_search = GridSearchCV(estimator=dummy_pipe, param_grid=param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")
print(f"Test Set Accuracy: {grid_search.score(X_test, y_test):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, grid_search.predict(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, grid_search.predict(X_train)):.4f}")
print(f"Test Precision: {precision_score(y_test, grid_search.predict(X_test), pos_label='no'):.4f}")
print(f"Test Recall: {recall_score(y_test, grid_search.predict(X_test), pos_label='no'):.4f}")

Best Parameters: {'dummy_model__strategy': 'most_frequent', 'scaler__with_mean': True}
Best Cross-Validation Score: 0.8876
Test Set Accuracy: 0.8869
Test Accuracy: 0.8869
Train Accuracy: 0.8876
Test Precision: 0.8869
Test Recall: 1.0000


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [39]:
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('dummy_model', LogisticRegression())
])

logreg = logreg_pipe.fit(X_train, y_train)
print(f"Test Accuracy: {accuracy_score(y_test, logreg.predict(X_test)):.4f}")
print(f"Train Accuracy: {accuracy_score(y_train, logreg.predict(X_train)):.4f}")

Test Accuracy: 0.9096
Train Accuracy: 0.9109


### Problem 9: Score the Model

What is the accuracy of your model?

In [41]:
#Test Accuracy: 0.9096
#Train Accuracy: 0.9109

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [42]:
# initialize dictionary to store model, results
results = {
    'Model':[],
    'Train Time':[],
    'Train Accuracy':[],
    'Test Accuracy':[]
}

# initialize dictionary for the models
models = {
    'KNN':KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM':SVC()
}

In [43]:
import time

for model_name, model in models.items():
    start_time = time.time()
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    results['Model'].append(model_name)
    results['Train Time'].append(train_time)
    results['Train Accuracy'].append(accuracy_score(y_train, pipeline.predict(X_train)))
    results['Test Accuracy'].append(accuracy_score(y_test, pipeline.predict(X_test)))

print(pd.DataFrame(results))

                 Model  Train Time  Train Accuracy  Test Accuracy
0                  KNN    0.058309        0.925820       0.897300
1        Decision Tree    0.229416        1.000000       0.886118
2  Logistic Regression    0.411797        0.910853       0.909586
3                  SVM   14.351263        0.923754       0.909071


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

##### Questions